# Projet drnouf — occupation du sol & topographie

Cartes produites à partir de trois fichiers locaux
(`notebooks/data_projet_drnouf/`) :

| Fichier | Rôle |
|---|---|
| `Zone d'étude.geojson` | emprise régionale à analyser |
| `Zone tampon localité 12 km.gpkg` | tampons autour des localités — couche `zones` (2 emprises dissoutes, `buffer_km` = 7,5) — cadrage rapproché |
| `Localités.geojson` | 10 localités (Jacqueville / Grand-Bassam) à situer |

Deux sorties :

1. **Occupation du sol** (Dynamic World 2025, via `cartograpy.data.Gee`) sur la zone d'étude — planche A4 avec grille de coordonnées.
2. **Topographie** (MNT SRTM ~30 m, via `cartograpy.data.DEM`) zoomée sur la zone tampon.

In [ ]:
import logging, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # cartograpy à la racine du dépôt
logging.basicConfig(level=logging.INFO, format="%(message)s")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")   # GEE_PROJECT pour Earth Engine

import cartograpy as cp
from cartograpy.data import Gee, DEM

DATA = Path("data_projet_drnouf")
OUT = Path("_cartes"); OUT.mkdir(exist_ok=True)


## 1. Charger les données

Tout passe par `cp.load` ; on ramène chaque couche en **EPSG:4326** (les
localités et les tampons sont livrés en UTM 30N). Le `.gpkg` est multi-couches :
on demande explicitement la couche `zones` via `layer=`.


In [ ]:
zone_etude = cp.load(DATA / "Zone d'étude.geojson").to_crs(4326)
tampon     = cp.load(DATA / "Zone tampon localité 12 km.gpkg", layer="zones").to_crs(4326)
localites  = cp.load(DATA / "Localités.geojson").to_crs(4326)

BBOX_ETUDE  = [float(x) for x in zone_etude.total_bounds]   # [W, S, E, N]
BBOX_TAMPON = [float(x) for x in tampon.total_bounds]
print("zone d'étude :", BBOX_ETUDE)
print("zone tampon  :", BBOX_TAMPON)
localites[["nom_canonique", "zone_tdr"]]


zone d'étude : [-5.619731503935415, 4.283466723722849, -2.677389907375435, 6.691252828934393]
zone tampon  : [-4.810766878931311, 5.073857840197228, -3.568095965326948, 5.391038930044812]


,nom_canonique,zone_tdr
0,Taboth,Jacqueville
1,Attoutou A / Attoutou,Jacqueville
2,Koko (Kokoh),Jacqueville
3,Téfredji,Jacqueville
4,Tiémé,Jacqueville
5,Grand-Jacques,Jacqueville
6,Quartier France,Grand-Bassam
7,Mondoukou,Grand-Bassam
8,Vitré Deux (Vitré II),Grand-Bassam
9,Azuretti,Grand-Bassam


## 2. Carte d'occupation du sol — Dynamic World 2025

`Gee` filtre `GOOGLE/DYNAMICWORLD/V1` sur l'année **2025** et l'emprise de
la zone d'étude, puis retient la **classe majoritaire** de l'année (bande
`label`, réducteur `mode`). Résolution ramenée à 100 m — suffisant à
l'échelle régionale.

In [ ]:
gee = Gee()
import ee

geom = ee.Geometry.Rectangle(BBOX_ETUDE)
dw = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
      .filterBounds(geom)
      .filterDate("2025-01-01", "2026-01-01")
      .select("label"))
print("images Dynamic World 2025 :", dw.size().getInfo())

dw_img = dw.reduce(ee.Reducer.mode()).rename("label")   # classe la plus fréquente sur l'année
lc_tif = gee.download(
    dw_img, bbox=BBOX_ETUDE, filename="drnouf_dynamicworld_2025.tif",
    scale=100, output_dir="_gee_cache",
)
lc_tif

In [ ]:
# Nomenclature Dynamic World : label -> (libellé, couleur officielle)
DW_CLASSES = {
    0: ("Eau",                "#419bdf"),
    1: ("Arbres",             "#397d49"),
    2: ("Herbe",              "#88b053"),
    3: ("Végétation inondée", "#7a87c6"),
    4: ("Cultures",           "#e49635"),
    5: ("Arbustes",           "#dfc35a"),
    6: ("Bâti",               "#c4281b"),
    7: ("Sol nu",             "#a59b8f"),
    8: ("Neige / glace",      "#b39fe1"),
}

src = cp.load(str(lc_tif))
lc = src.read(1)
left, bottom, right, top = src.bounds
src.close()

present = [code for code in DW_CLASSES if code in np.unique(lc)]
# Remap des codes présents vers 0..n-1 ; le reste -> NaN (transparent)
remap = np.full(lc.shape, np.nan)
for i, code in enumerate(present):
    remap[lc == code] = i

lc_cmap = ListedColormap([DW_CLASSES[code][1] for code in present])
handles = [Patch(facecolor=DW_CLASSES[code][1], edgecolor="none",
                 label=DW_CLASSES[code][0]) for code in present]
handles.append(Patch(facecolor="none", edgecolor="#ff2d95", linewidth=2,
                     linestyle="--", label="Zone tampon"))
handles.append(Line2D([], [], marker="o", color="none", markerfacecolor="black",
                      markeredgecolor="white", markersize=7, label="Localités"))
present

In [ ]:
m = cp.Map(figsize={"paper": "A4", "orientation": "landscape"},
           title="Occupation du sol 2025 — zone d'étude (Dynamic World)", dpi=200)
m.add_raster(
    raster_array=remap, extent=[left, right, bottom, top],
    cmap=lc_cmap, vmin=0, vmax=len(present) - 1,
    alpha=1.0, show_colorbar=False,
)
m.add_polygons(zone_etude, facecolor="none", edge_color="black", linewidth=1.2, alpha=1)
m.add_polygons(tampon, facecolor="none", edge_color="#ff2d95", linewidth=2,
               linestyle="--", alpha=1)
m.add_points(localites, color="black", size=30, edge_color="white", linewidth=0.7)
m.add_gridlines(color="gray", linestyle=":", alpha=0.6, fontsize=8)
m.set_extent([BBOX_ETUDE[0], BBOX_ETUDE[2], BBOX_ETUDE[1], BBOX_ETUDE[3]])
m.add_scale_bar()
m.add_north_arrow()
m.ax.legend(handles=handles, title="Classe Dynamic World",
            loc="lower right", fontsize=8, title_fontsize=9, framealpha=0.9)
m.save(OUT / "drnouf_occupation_sol_2025.png", dpi=200)
m.show()

## 3. Carte topographique — MNT SRTM zoomée sur la zone tampon

`DEM.download` mosaïque les tuiles SRTM couvrant la zone tampon puis découpe
à son emprise. Les valeurs SRTM aberrantes sur l'eau (voids) sont masquées.


In [ ]:
dem = DEM(work_dir="_dem_cache")
dem_tif = dem.download(tuple(BBOX_TAMPON), out_tif="_dem_cache/drnouf_tampon.tif")

src = cp.load(str(dem_tif))
elev = src.read(1).astype("float32")
d_left, d_bottom, d_right, d_top = src.bounds
src.close()

elev[elev <= -100] = np.nan          # voids SRTM sur l'océan / la lagune
hs = DEM.hillshade(str(dem_tif))
hs = np.where(np.isnan(elev), np.nan, hs)
float(np.nanmin(elev)), float(np.nanmax(elev))



Téléchargement DEM:   0%|          | 0/2 [00:00<?, ?tuile/s]


Téléchargement DEM:  50%|█████     | 1/2 [00:04<00:04,  4.11s/tuile]


Téléchargement DEM: 100%|██████████| 2/2 [00:04<00:00,  1.80s/tuile]


Téléchargement DEM: 100%|██████████| 2/2 [00:04<00:00,  2.15s/tuile]

(-99.0, 130.0)

In [ ]:
m = cp.Map(figsize=(12, 7), title="", basemap=False, verbose=False)
m.add_raster(
    raster_array=elev, extent=[d_left, d_right, d_bottom, d_top],
    cmap="terrain", alpha=1.0, vmin=0, title="Altitude (m)",
)
m.add_raster(
    raster_array=hs, extent=[d_left, d_right, d_bottom, d_top],
    cmap="gray", alpha=0.35, show_colorbar=False,
)
m.add_polygons(tampon, facecolor="none", edge_color="#ff2d95", linewidth=2.2,
               linestyle="--", alpha=1)
m.add_points(localites, color="black", size=32, edge_color="white", linewidth=0.7)
m.add_custom_labels(localites, "{nom_canonique}", dy=0.006, fontsize=8, color="black")
m.set_extent([BBOX_TAMPON[0], BBOX_TAMPON[2], BBOX_TAMPON[1], BBOX_TAMPON[3]])
m.add_scale_bar()
m.add_north_arrow()
m.ax.legend(handles=[
    Patch(facecolor="none", edgecolor="#ff2d95", linewidth=2, linestyle="--",
          label="Zone tampon"),
    Line2D([], [], marker="o", color="none", markerfacecolor="black",
           markeredgecolor="white", markersize=7, label="Localités"),
], loc="lower right", fontsize=9, framealpha=0.9)
m.save(OUT / "drnouf_topographie_tampon.png", dpi=200)
m.show()


NameError: name 'cp' is not defined

Les deux PNG sont enregistrés dans `notebooks/_cartes/` :
`drnouf_occupation_sol_2025.png` et `drnouf_topographie_tampon.png`.